# 🚀 MotionAI Coach – Google Colab GPU Pose Server

Notebook này triển khai **Server AI trích xuất Pose 3D từ xa** chạy trên GPU của Google Colab phục vụ chấm điểm động tác cho dự án **MotionAI Coach (BTL IT4409)**.

> **LƯU Ý QUAN TRỌNG:**
> Hãy vào menu **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Bấm **Save** trước khi chạy.


### Bước 1: Kiểm tra phần cứng GPU NVIDIA

In [ ]:
!nvidia-smi

### Bước 2: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok httpx ultralytics opencv-python-headless scipy numpy

### Bước 3: Đồng bộ mã nguồn MotionAI từ GitHub

In [ ]:
import os
if os.path.exists('/content/projectCNWed'):
    !cd /content/projectCNWed && git pull
else:
    !git clone https://github.com/trunghieu714/projectCNWed.git /content/projectCNWed
print('Đồng bộ mã nguồn thành công!')

### Bước 4: Lựa chọn mô hình trích xuất Pose
- Mặc định notebook sử dụng **YOLO Pose GPU** (trích xuất cực nhanh, ổn định).
- Nếu bạn muốn chạy mô hình 3D chuyên biệt **GVHMR / WHAM**, hãy đặt `USE_GVHMR = True`.

In [ ]:
USE_GVHMR = False  # Đổi thành True nếu bạn muốn build GVHMR 3D từ slide đề tài

if USE_GVHMR:
    print('Đang thiết lập mô hình GVHMR 3D...')
    if not os.path.exists('/content/GVHMR'):
        !git clone https://github.com/doantrunghieu08/GVHMR-Demo.git /content/GVHMR
        %cd /content/GVHMR
        !pip install -r requirements.txt
    os.environ['POSE_COMMAND'] = 'python tools/demo/demo.py --video={video} --output_root={out_dir} -s'
    os.environ['POSE_SOURCE'] = 'gvhmr'
else:
    print('Sử dụng YOLO Pose GPU (yolo11x-pose) làm backend trích xuất...')
    os.environ['POSE_COMMAND'] = ''
    os.environ['POSE_SOURCE'] = 'yolo'
    os.environ['YOLO_POSE_WEIGHTS'] = 'yolo11x-pose.pt'

### Bước 5: Khởi động Pose Server ngầm trên cổng 8100

In [ ]:
import subprocess, time, httpx

# Tắt tiến trình cũ nếu có
!fuser -k 8100/tcp || true

cmd = 'PYTHONPATH=/content/projectCNWed/backend python3 -m ai_engine.remote_server.pose_server'
server_process = subprocess.Popen(cmd, shell=True)
print('Đang khởi động Server...')
time.sleep(4)

try:
    res = httpx.get('http://127.0.0.1:8100/', timeout=5)
    print('✅ Pose Server đã chạy online trên cổng 8100:', res.json())
except Exception as e:
    print('❌ Lỗi khởi động:', e)

### Bước 6: Mở Tunnel Cloudflare (Miễn phí 100%, KHÔNG CẦN đăng ký tài khoản)

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

import subprocess, re, time
print('Đang tạo đường hầm kết nối Cloudflare...')
p = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8100'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for _ in range(40):
    line = p.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print('=' * 75)
    print('🎉 CHÚC MỪNG! SERVER AI GPU ĐÃ SẴN SÀNG KẾT NỐI!')
    print(f'👉 PUBLIC URL: {tunnel_url}')
    print('=' * 75)
    print('\n📋 HÃY COPY DÒNG DƯỚI ĐÂY DÁN VÀO FILE backend/.env TRÊN MÁY BẠN:')
    print(f'AI_REMOTE_URL={tunnel_url}')
    print('AI_ESTIMATOR_TYPE=remote,precomputed,yolo,mock')
    print('=' * 75)
else:
    print('Không lấy được URL tự động, vui lòng thử cell Ngrok bên dưới.')

### (Tùy chọn) Bước 6B: Dùng Ngrok Tunnel (Nếu bạn có tài khoản Ngrok)

In [ ]:
from pyngrok import ngrok

NGROK_TOKEN = ''  # Dán token ngrok của bạn vào đây nếu muốn dùng ngrok

if NGROK_TOKEN.strip():
    ngrok.set_auth_token(NGROK_TOKEN.strip())
    public_url = ngrok.connect(8100)
    print(f'👉 Ngrok URL: {public_url.public_url}')
    print(f'AI_REMOTE_URL={public_url.public_url}')
else:
    print('Bỏ qua bước này nếu bạn đã dùng Cloudflared ở bước 6 thành công.')